# Đề 12 – Kaggle GPU (T4): DINOv3 + SAM 3.1 region/re-rank, benchmark 70 video
Trọng số lấy **offline** từ dataset private `holilihu/de7-weights` (DINOv3 ViT-S/16 dạng `model__file`, SAM 3.1 chia `sam31_part_NN` + sha256). Không tải từ HF trong kernel (`HF_HUB_OFFLINE=1`). Thiếu trọng số → ghi `results/STATUS.txt` và dừng. CLIP baseline không chạy ở đây (không có trong dataset; số CLIP lấy từ CPU box).

In [ ]:
import os, sys, io, base64, tarfile, json, time
os.environ['HF_HUB_OFFLINE']='1'; os.environ['TRANSFORMERS_OFFLINE']='1'
os.makedirs('/kaggle/working/de12', exist_ok=True); os.chdir('/kaggle/working/de12')
CODE = 'H4sIAH9622oC/+19y5IbR5KgzviKmJSmmVkFJJB4VRElcMSXKK5EsVRFSd0GYrITQAJIFpCZlZlAVXWx1mZtzdZsbzs7573s2OxhbQ97W7PWYQ8ztv+h/pL1R0Q+8GBRGorqaQI0FpCR8fTw8HD3cPcwq2b1s2Pn8gvXGbnRR7/Ip8afbd+1WqOZ/cZ0q1a36h+Jy4/ew2cRJ04EzX/0YX7qh2KeeHO3ax3crR+0D1ututls1FvNxmHpo93nL/6zjF0nGk6rv2QbuKgPWi38tg5atfx3uuatVr3dbkNqi9b/Qf3gI9F6n+t/ELyZ3Nz2/t/ox/yzoP+Ndfpv7ej/e6H/B3n63zpsWy2zddA8PLB25P9Dov+27fleYttmePWLrP92s/kW9L9ltRq4/hu1euMjUXuf639H/3f8/47/P2y1a+Zhs9Y8bDZ3G8AHRP+HQeT+ErT/dvoPS906yPj/tgXrv15vtHf0/318NE37zhu5gfDmzsStBoNX7jARjBQdMY6cuRuLyj2hjxezWYWexb6I3IkX+IZw5wN3NPL8CeX5/P7T01P8EbuTuesnsQm1l7x5GESJeBUHflkEcVlMZsGgLJDoqHfDZb0s/MU8vBJOLPywLMaOF0PWJIBulMZRMBfHT78SMvtT7CqnJpHjx+MgmrtRrF7fXyTBs2Dkzsr08zgKhm4cBxE/UuE0rVT67umjx8/tx799IbpC18x52NTKQjOdpUff82DJ32f8feEO5ppRKn0sKiufFFgrn9LIHQtn6sRTfRwZnZKAzwQag0GbkRt7f3B1/DlcJg+DWRBBpjK9e/j8q+cn9oMnJ/UnJ/d/Z5SFflgWh/Dt+YkbhcHMSWAOupj16dcvHp/Y908e3zeo+shNFpEv9Im4Jybm3HV83TDMMZRIXPhZoi7FzjycuTb3Wg+dZApgD+OuZdbKYuSOFmG3JrsL0/it7yGYuRRMuMlZ7tU6Ij7zQuHw8MXFNIhd0W5WBl4inC9g1MKLxYWXTD1f/J7K/F7Au1gEY5FMXTFz4kScuWHCFVB7OmCkIwZBFAUX7kjQTD/yJt+Ghil+57mzUSz0xI7dYRnRwqQpNQjZsDTV2BVfBz7XNnRCCW7C9IdOCNBxacTGkVjCmPG1E5oTN6GpeHj/2D4+eX5sf358aoggEvWWWaOq4sQF5PQg/9y51K2yiIKFP9KpjipCz4D54awXU2/mihfRwmUY4ic4AxBHsrUIOC7dSN95Y+EHCWTpiAG8Osu/8MRfU9Oi2xW1rDr8TKG2FLkwL4FYuDOYhRQCuar4teOPGEwwN9gq5qREwC1dn4q/6tJrw4wXc0Ad8WmXCxbbpirFfldYR2IY+InnL4rtyZmYFhKvcAKhXJVAX+bVbOIcO1HkXN22GE6ePDByUKPm1TQDTGeuE7v65gXKVGvLCp1E3siWOXRvPimLmbsEMHZ1oE0NmNZg6UYzBxaFWW9l6+KRmwDBDCIgja4rWxBhFIRB7MzijioVwpIR88Us8Srx0AHEwOZMcUILNRYzDwAFC0K/rJWvauVLq3xlZQj9fVl8AWCETplILo5gZVy6iLS9Pr2HhSl8mDrZ42yShhcAPcSQ7wHaPtYCX0cihOQQk4cXYk/1kHKmT2kVWHeEdQOhnbi6bxRRAF8Pt7/GD3XWBBC4sFJ0XDc1omH6cG94UQkvcMnkUqO94bQSTinV8/XvOVUf7lsG5N+X+eHNF/JNRG+m+1jGKNA/arlUGgIaxuL5918t6yc8v+nsPQ+RhjozEUDvKstg6AzkFujD9pDNo9CfHH8L1Q6DOWxssBSAEE2CYDKDHfNitqxXBoB1FaAow6nVrrh+DBvjzE0nEPFLCdh67M7GsNsBretqt9eh4SYYnnUPkSYvvSEUGoYLLQfnrfvgc6w1t/vR8+dB9JwGyIgLo08rwo6ZSCqLBWlp2iHANHI83x3p2HUgnJR9rrKv17u5nJkEOg/EMN2lM8tRQKoQKCR9Qx6oGoeuBk75PiOOwPQDexKl5JPBC8tqloIX1koORJdQFY9OJzYn7tICZzSxYU+EYcZdLUw06p/qQNa18Zz2EB6ySXXYzPu4kR56l+7MhrEsoOLLnpZ/1vpGr9ZP6wG+B3ZZWH4jqA0rNeOpE8KKHrtOEqs04AowVcfce5g56wggZ9aPDFMRzCMP6ZBONQHh9ibzwAMIFZqXdEOWh8fVgmXqARYS4mMxvBxeXUxxeYcODHVUic8XTuQCsQ8i2IN7tbKVVX4qd0VFpgBFojyVUuTiFdYHXTdxchnWZ4bp+dANoBNJgLRQX6Ejw0sgT1dlMQBoDJBy6TSU3qs+kKxTIy1VKAS0VAAxFUBNxZUlewfPw8vK4KJaz8jO8KoymHIC0BXVf4AB5t1XefOvLHx1tU+lVnfYS6tyWQPGy2rTjnplVa74sYPwSAkh0q7LGrNz+pX6cWmpFMvIbXOSoEVIzjZsbAoV13c2pn2nHrJ6jzw/Y4ZTEvgCF9TSi2HF8h7mni88QF7g33FHevT06+fLxnfeiyLfjGvOge0spkEBSES93ryE/8DbAXsI4oPc2L92E4EMaDVORm+ih1hHF8rnJp5Qg+ruCt73GG+hNkjxQ5P5hV7NbB62gO0ym602fdXafUWfoNWVvPX6XcwETfFXq/8GIoK0YjOdKBIXaADEyOGZ3tOxrZhb8+YmsEWwoSa6BmwLEBfJ7+vp0MrZKA3FCj14+tXTrx/fPwG+qwqsJ/CeopINHRPToeF68ua4oLiv/TWUuS6Sow6LVCaPRb80aeOATc7FhdAoC1griO2j5Cp0u5x3PAucpFE3btRW+liiW4pDw5kXdmgPdbwqPlSWXpLbzxp18RqmcAJv6h21ccrn4raHGPRagEQZLCHn2Bm6gyA4q3JCJZ7D9LCEQCkN4ZogiRSzNbDxGOpSW09lthxZ7cO7c+R2J07ijjrCd10gYF98DuA4c32QaGdAfWDLFc5wCLKI4w9zAsXW7ftNI9bSDRuZ61W0xvKIN/B1lN/wuAjKHLo2XIwcDSkKTwM+ml5sO0vHmznAGugGs/nMEBTrx305lYO378TpTrd5LwaiDPVoCFbqiK5++tRzcwbiWaSTiJS9ASkK5MxIl5sMrICxB5utNseu2IhXKEhrhur9cqblZZ1cy6rp4l6QRFedjFdZF+k3j7ZQhXuJkwzLDfmkxyBlRrkaNxBLGCLshkwKkSmUNJCRKMkIKGxLEeFTQMxn1iiOdKXTb9Pft+d2mFAN4i6s05y8uUjW918vY9drKOT4kicysPzKvruJccKvntfx9gdxfzNtPKJy1yDJLgtIRs0DP7fEHlyaXuLOYx3IysoGeisGKLmXMew2/stEKdSeekCxfBuodOKuVeaD3Ab1bcNcfzEnyRBE8MgmghEDCtcI72s5zAAxXDSFysmkJV5rbEx8La1oJ9F7014H6gJI0g8LSBH2ptNntY1lwJuKRfj38KtTeEvbH9GYIkojhm1sS44GBmcz44qM3iJCnc+bAbcu67OKAlDdj4lE6mOpojNf0PwbHWoR0CUIZgArwL9wkRTqgSTFAknkBkFh4Q9ZDgNUj4DG4x455n1Hx83Im3cBBibQOd0wSU+orzNHsPHCjAFMgSYnrg4NGbATI73RNbmFbVHcAaq5l5u1AoOFNxvZlEFfovbIHnnA2kDl/IOI2q0bQapWk/oFEPxA/IcXs2DSDSPg91Y1bkFszp0zYM2jWE8bcy8BsezgrItaJVhlwPYBtNVurFNfGCwwU0rZgWsoduYNbUVilIp/E98pvYeSHE/vP2tIUfmIhGAki1maJERFlJPZ8mK2FPS60E9e/yvdIrFX421Az+tfVjMStDKdFje/dIdA8uawZmE7RkSsEa0r0/8aK5hN/CO7i8SHKA8gKvAA+phSxrRlzYKBiX90gDvqBc1XAbD7uQnX9nDDgm6N1aZnAgrHqNfUUw2ykZfKoe6E5FBqsqBrXRJGyClflXQAjDgsHAsUxuFca9QR4N1U7xC9cJfQl7A6tATesB4yYU2VhsQTEjVqD3db2IzgGWF3018ltzhxGygH9H9Akh+81ovi9Fp/1YpG+QgT9IEB6DlXydkA5iBU9eRTX3ad0xJ6Vh3nyc96PrgpUiOce1U7YJdOvaA2AR1QJTg/ApzIdIO/ZR49Tx+wCigwwh0yhyrAaie1FF9oFwIg6BkoZddy4PBGcUHG8HibBcETi7pAr9wIW8S+EQ4BqaWB9hG58VdfMdwakIJ2UytohWmHHsUGK3/XFa3eCHdbOjAxnyKhevromRPquYTPZ07y9Fj/LSsbQHhFSEExE4R6G1HYhvr13/bgL/QE28rQmGq5iGCvllQQygH6ateSKt1Ur3EENybllD3HXRZhcs1MH04yn8dogPfwRNgvIYnP8hek0ZKAFPouSwzGLLjCtSWpWvERgULwhFTqF54GpOtgBAsBBWlZ3A5hL8q/98dVzFKXvAceTZkj2Fb067096juyq1A7dhy+bsok2+j5UWO6iQURJy40uR0BPdepAlJ+0LZEjxu3HSbAW2V2equErA0iCA+bNwa5BWwUNZCFpxHCHjjS1UDS0vmhGKlaD5eRZB1GPYZFv1gtbz6p5gEwPt2I0mI0+f0VqYLxFblDRi48B5G4VezX9VmKVbwYt6xEXCSKMtIWGW+vCGRYBU4Gf6Yv5M28m9Z/1rVqpHz1u/VaDXeaCJiniRMSlMviHLBxFeAgMWIl8uAUJFl5IPAaxAHoXarvB/YbNvf0nBRoPNHEMp5FJ2WgauXQdc7spBzjuXwZiOBNPxVH8XMOIMQO4Nj5O3eQQxulmiSdthGY2IFLhzHXN4U9KqVwvTXA9rFyHA7twTgAuQf3MLG/siE8KounCmVgintE10wJ5HPWnyEwy6tZ/CRInJlhrJ0rwCokXeEfvFB/RNq4p/DXeJMsAHjae9U/EmfuFSoJp7m9Zkp7zEamFnMTY+szkLBtcY9+9+BdH1rtZE9YMbOmKDJBtbhF5SqekKo4xRYCYvawOk2W2VIKHVgxSB77q4wtl1a4omNHytiK0euc9fksd2uWVEXzDHMIkLiAwQPR8Uo4o1fOEJV8UzyM1WPEV0ZC8XqUvP60i8VRFRmkaHqkfglCS9aoYnl+ThF0cGUvM0zDqUSWJ8EjbBRSDQVoJQF2qATgSiK1ipi9B5Kb0pXCYKig3F3cycrhF2SfxlQr1qNqTeE4jU3k+XRYBMNFlD8aznFq2ABWMY07qxwSFkJVbgLMAfzuaS5t3p92EcTr6JjLAqzFJoRD7KJcBDcNsAuezEU4QjaB0rrQHSYBXegbYFh3YLyFvMed7RCIFPQgwdjUR9qfFWMGENSI+KCGEH67xIfhL+6FfOD+dhBYKWeWo+4bGlW7IPPcmKWM6607c+aDkSMuO6JymcLB2Jmc7ew/d/afv6L9Z6t598A8qLWtdmNn//kh2X9GLvKkv4wF6C32//VmuyntP9sHtYM62n+2Dnb2n+/L/pO1dbBTVxAF0FAJTfOCsPJlyvl1iEtyHRBXh8AJecioqJeo1aygWCmQkUR2gaWfMurThTQoTS1pSvrp/WeigSckT46/PSI7KHx4ePwt2n6iXnlU8XxDCrWyTdQtoqKCLAaJVXX9IQghIPsiYxa5zJDSW8miOnEJj/eDyJt4vjOjw34bK7KHQeyB1JaTxtwK8yeZQLaPOiE0c8xZsKLtKpLK8pq96hYLVWLNSbwlgEh9IltaJpI/3WKcmGVFphXtRFZtE4+fn9rPTh8/hKrEnrBgHUFOMjAs2BcerdjF5XiyNxvebbS7I1H7LGddKIeIiKPHUmNRFjnFKQnWPPtupERskiWBDwSxH2Rr/BEsEhK3V4VqUurm1HNHbyX5cj9y0m9ebkhfS9mUpXnsGPdIduIc2vJt7DkWylSCJGdgVXl9OuTIzfM2JXKciqMGPUjWul9Q90FlMCLs1AZdnzLgUQBFo8+sm/td0otRpjXDUkrdUOXjPEgIYuNIKXEzLTCVJqsKb44deCw+E+cgFx8JtEdCgxV8YTrRBBedUWgd36Clzj0ccyrxxEV5R0rUnNVgkYdsepShj1GWeOaO+PwjnVMW7t4gVOTlEMgPYjPiW78M8g9XKZWDq2pglH4U3UIVJIP55i+KL9r5/+78f/P+X+1a27zbPrRau/APHxT/nz8GftdSwC38f7PezOI/WI0D9P89gAW54//fD/9PDLlprTkvsNF74M+uyOOAWPRo4YsL4HmfP39WOfNmM+CZ9X9/YNbFkwfi5PQUzxX9hCzkiXf40plMZi5y+sBuf43mUh0x8RIxnCG3puz2gH3PkFD85jci9EJBNiazmai4wiyXvvjcfvH8y8dfsxSARnoxWl5lpn9UtFp8Ni1T3D9+CjJJBIMLA3Hy+P6jZ48F2zLxmSRw+qeQNTNsBc7fvSSBZR4mhln6Noa8nYIhiGmaOVMOMq4w0O2NuTMQnrDfykYmBsEjGU5RrELxhOtFSWIKEpM8PhuhGCHBzwfqKHZsESrk8WBmjrH1hFC21dU1lsK0spF5E4B0YzPrVTMbBVtF9ARxh2chcK8JiSCrp1zUMQIw2/QRbNxI9ZNBRQSFDY4o17bSZi5jZlqnDFHyU5PxlPyS7IeOlMnkO7GdTCW9o1UYoFC1kgINAafv+ksvCnxyHsOl1LAffnn8Ilcno1l3C1j0vFcHGgSRFBHM7S8+7642KOWClfNUKuL546B4uIdHAcXyHTQiuwiis5gFGZyCMCEBmleLTQ5K4cy9hHRgh50zV9wZSQcnc+8OKu7JHJqANPfi+ZoZ2vAstW6j4+aVLqAUHtqzYMi+i3LQF643mSaxjaRG2TZRRcMzBisfJANenhVPMuIRnSSDeBvOYOHrWtpZMjFFI4COWBbMDqFKeT5EJ38mHXqwHU+u9IpRIsp3jKwEbDIitNFtQo9HIMklEfzsfg4kc8XOdH1+NIBGssmSZwVQBtlCcHdsBLu0eYhH+AZAHwNBkWmRKZ8pI75f+O5lCGXdUZolS0pzqVJsnITGEYWKep1WvwgGslRjJCdE7WjllSGiQQ0g+xC9eEZrK4BBeIsDENIANDfLr3xp2naU5mASWnAYoiWaUVdyHEpp3BvchDpSD8JOpDnbCZzk1PwVeoIqGF65ZA61xbslZPVA1s3OqvnjWp2439icm4xH3C79VfS7G64fjQ/KrIfAo3Goks6g0WYzdYQhE0UpAufTjc6W88iugmLRSwVBuQDsIHcVacFL62hgGGSUenl1eSXIcBT4haUbeeMrsi8GGrFmodkbUHlbnrTKM8GIXSygbOzy6u9JU+mzfn8nhOzk/538/77l/+ZO/v/Q5H/Jg/0iB4C3yP+tptVekf8BEw928v97kv+fj8fosqnYcBbQpOQ+chLgUNFAzA/QXW4UXPjI7cUgz58uQhQEkMOpwLtKnFxBiXGAsiAUYPcZMiwFoXoP5LyxK92EDBA6ri6mbuSKhY+SY/WMmqt6PvmLVATGKhFjbwYsFrLHI3HnU+IB79n2p5h8747QyfdvCozNzJsuqiP3oCJHQMJ45FZATiYf+hGbsalGUAQCJrd6UZV1VsXSc0R8NQcwnMXYfBzOUEuR8uQxtI/9uAfMeZTYX38Nze+LQEUPUC/jqVNvtc3kMrlDfcjZm2/vAxUFgaukjiNRsGA9h3RyrJJ8ZtkoO8gH9Rs9TtAbaBqzhRoGVAkmExfDe1yhr03u/JLj7uApJsfiwaghMw9Zyeki8WbIhPF53sgDvhH4MZ2mI+5qhelBgQ16n6WmYyk6tDAziDBhro45T57TrmL+woxpzpwwyHoXW76p7u1V9zTs2XARxd5SMoh5i18vxir1UJ45MYN9rfHwQajpodW2zBu5M/zWMYYLNWDcCD19C6ImeeeGRrW+t1evdUxrfCOePTC0rJPU/X5m2DgjZ3ZktWWD/Q4ZgWtCIDSMNWceBNaqJw9l+phxfg3PSwXJgtrPeHg0wV4TJcPC6Ztm2+SWOShy/nM+KB6YhOo65kJx+YgjE2jX2M+b6vX8Rjsq9H+0sfOrQWzWrLFHUNf4RjNI8pUrDach90pCYW3pUTouu7hoU7oJHnM6iDVJL6FHmm7uG7xi9ZejfeMTrbwJWgVwzTvcVt4aNT3QBQjIMiBbmpMoWIS6hfFIcsaqYeZthMAiZ04Q5WKOooA1r9mnjsgwOwM6lrkBiqCV3gxXKLcizxHVIBP/EVoAaxcDzUDd4sVmB58wJ4aFMUwOUwJzGIRXCNlg8IodBtAVI4K6KHaFJT79VNSbRt7NFFcZdZsIkx05F2T9Cr3IBOopuhQUVvlIDVZSzr2i5wuWQKqWUZKV0To+Ao56CC/ROFxaPUisxjgWJqpnQkiSfltHZC4uaZ/J7a4EjliFYiShON7iJjUjzwSY1kjnE+iOGHNHFKhA4tUQ+afq0BvKGKuKgQyG3CtYYgREfWpO3cuRN3FRgkdjfBw4DAQJjSqDc4eFBKzLa6rszmpld/pqlU3HGQW+Xkfu22lybm/fQJ33i+UYq4uF1i2SccK3+OFtWn5agaHQDEN6iIUeaQ6lJcDeGbr6X8IS5vMIcnWA0Xuko9fJQWuUurFjDko6wzWdqVpxH5XzxjtyF1thZ/hlA1WNHHFAI4/gwjvyXsVf9mBToQEWKtMuLl/jz7RUPV+mrpyMaIC9cIUIkitf5pSoa0hAMGJdmEyhZB9mpVdUheKwCrrQtLCWrWKjr0DQ0zIuhPCSAZvrB3SMHDeVZU1EJXqm3f8b6zUVfp1qmDeTYjUVa/Dnxm9t+w1Vpo6wCJdC9VI7Ba18gCLvzv57Z/+dt/9uWRjf7rB20GztFEAfwCceAneUxH8m93+0D6xam+7/aLd393/s9P87+v+e9f+tptUwW4f1g9bdHfn/kOg/hoIYeSArOxRi4Z2eA9yi/28Azin63z5oIv1vtnb3f7w3/f9XDrqGD1x/OJ07EUbtcmbZo9CtGjt0xGVxvnAjD2TNBbzECGIjFPbx6P5KPLr/3dNTUYeZFXsUyWxPxC7k94cu6k4o4veFX8pwjCsVertGuvID1YopHmV54mARoQ0dWi+QDl22c/r825OHj0/N+QgPGkIP3iWBCuIXFzB5PjJWvXhYE67U3uTNwyrxyC3F0ZCMJ8riEcW+cxKnOnKWXmwTSFCkXk2zZwjB9A31UCvltaYcOeSmyiPU1nSoGyIRQRHoiipS3aPI9FLdJ3W0uSqr12ui79iQutuCytCQZmtKB6veO4OYVONjg+LNYofOsUM9Tc45Dk/+tKdONNL66325Pv9ZTcqRYums9VebwaG6sCf1SAWdpf6Kps4oAebFSjMIJR/dVMm859QFSocoWiUMxYMabVVleCSgnfmoWDpDtzT/EWIb2//k3ILOpTJ6uT5DGU7a1/ENTWdpu253aaz4F0l1XQEi0LN/d/z4CY0trjYPa2H1GhV05qtwQsFcLqK8c9n3Eekol+XVJHuMy2yo72nQqyVOAkZH1Q9b8BdqzWnIM51TtBImYS5b8uYEn5Ba58hBOoYoBfSeywBEnTpFPtKh6rKARqQtZu5CBG9eaD5rH6os+rJJjb/SOI21l3viGhLceIghlGPj5mW8/7L3FSDey/5LXe/9rdHfN14aaLqG88zzqJT3Yw2Dj67Ok3gt4Bf+zZT+dFbA/b4Tu+4aWboD+TWjRDikwagwVm51C32SIYQksLSP8wSQkUfoOfJKuGu89O/7gDdkSBl3xMOH4sHvRNOsqaxBNHF8gGWER7WfUxykjowVPkUjWXaLlARWNSPJ7ECRc/13weLFYuBWj1208KoiziyD6p/+7n+QOxYNwxssCmUoYry06AYccMUoIATPZ1ek/qX/0n8tN4LX2X7xeq1/ryFfpVJJ/7/0Ndh5NPhiFTFA1pAJRontFNHusbBa8otRklOD1LBEkz/Q2BM7/d9O/5e7//HuwWHdbFi1ttVo7wTAD0j+y/m3vHMbsFvkP6tea6n7Hxss/zXaDWsn/72Pj5SKQIAJnSjGePTo4oEOMPBDWtgAyxIlGCx69SB227Pi7G0bjyZtmy5mKUSdpXANBX8hQr0SRUNQnTHvR5MFxmM4xic6u3dCClrpyBe6VqmkMlWENxdEyj18Y96AzJeKGUsb8imHE2kBckuA381tYSy3sqCwnuTcntVmmbWNraqAmLl2ZZzg4TTA2zG6Pc0PfJI05QsZQ7csY/z2N3eFo2rKzshow9wATOrUnYVdja/qmjrzOXuq4R0kwWxUFrVuMB7TEHFmQpNmBmvHMMB5rzjHVFoCxyT52TFl1E/HpICfjpnG+eRgx47JEXB3G82O/9vxf38O/J+M/1Wv7+J/fWD8H7riLZzE/RXu/261WrWV+F8Nq9Xa8X/vSf8vVaqkUO1guHlyctQ6gj12tQmZUatYoWUZJ7RMMUJv+jd9U5xwOGO6xVCGvs1iaZml+2lQLg/d3GeAa/h7LLxE3TcYA3vx5AVfbYrHD/p+tSKSYGaQ6ugKL5FCZgvDDVBEMMirEgK6q6d0Ql7JsThx0cPzs2ur3CpbtRuhO/5VrlGfI5uhAe/9488siuvs+CoW9wyt9YdX+fOCjDPmIwLJH6tQYLkoYL8Qy7waAeBIbOWkVczwn8NGExP3dlx0diLwdpx0gaPl2HDMCZYpENjmRmDyt7DPNbO1nbvf/AZjvzmR62xnyBUfjD7oKmidOu0ie+C5E58JrAOj29F1JeJT4JO9eEuTHFkqP3JNtYG8O9lDdlTQPULKeo1bWo3Dod0i92QNbOHSv8FjinORnumsRIN3TDlQju92zj73OFSMZzbkq24+RXZeQrF/RJf9KWyDCgh5MMY4iEU5l4RtSJ5rUblRX98UBUQGn0Jsfiph4DsVvhlNeFWu9I4Q6WOybX3k7xg5ErK2fGLJna3Wm7s75afdmyKrL96b8s13ZfHNi/Sikuyc7RtuAu8L6UqPdJqcgiE0grcsznuSQvcNY+WCuSORrMbL++Y7dbhxmguIhy9epBfxFOOe8fkbBX6HjrEUFys/kbI6gzzBSbM6NJAWf1k1/L45EveP2TP+CAmq/HX8ZTFm97lX5pFnF2R8kzvzWhlH3i1ATU9nJdYChpa26TiIwhGeosSpRFKOa4JTokL9YdzBNOLgN9/1zr3+yn1da9XjbG6KGri1Ghi9gvE3L/AVx5UsAltlXsTuCCnqjAIEkA//SmyBDaEH1REY20OnN49MioAFjJlgnEG+S1qGusf2yL1ikoYlRITPghRyZJCe3O853rkKbL4PsAUKrbJw4j3KoLJXOEvB1DoLCDBTgJnnAzgCWmK3kNbp8+KpI/lYnHTESe+snx7XUdBA2OB1qBCj4eePCpOAr2ydxBSn8PjL9VLFkUOB4tgnhbHTaxWysQAM7B5BXME51wuKrt8VNSThiJw1eZN4ehtbma9XziYLhmKshaCPOlwTX7aNVcEvSqkKmPZ9YWVNwupTQ8WMVbp6AU/DVPcQY9ObkmO+zQFX897e9Vg7+ez67Ca9KYVh5Yd8G9kSD8waxXvcTtJ73DCkCPFUW0pDv7g8XvwThJZNwPR8+8mLLSWOv0wLzGM6+r9Kc2LE071idlhumJ9ufvFtxaVwBJRvZEgXPhvkdZsO3uDdBPiHTv4yGL5ZBio4NUd4+qvuhKF7lOieDDrWpQtsYnVJjFR+4YEu39HiA3/x56vi2tl/7uw/d/Efd/qfSbhgY7ZfQAF0m/1no9Fajf/YqO3O/96X/gejPML+GS8il4Of6zL4wzHsscA+vGgaHXXFKQcslEdEwWIyDXGz45DuZ+4VX3slBVhgSZWkHIcuMLclCg7/3QnkxoALudB8KFnO3XkQXdnOjMLDARMoyDgNNUZsPZWhKO3R4sR1Zqh/GWC4CZTb82obVtawjkaae6aKG2y39N6ON9W1aOXiLZAljE6hohhuClCITAxdMAzzo5X4euhcZrzPleOo2dQ5DP9OMpkGcKIb9JbEZ0GB9NZv28YoWzAhtn1TWvK9hWt2hVvMW1ctpzA8XImHwndVZrHBqGYZ6YtvJ1+7/xKPQIEpP+kBoybv+EMWFDk1fjQoEgUiDN4PJV005S2Xb8Ybna5N39tr1PCGv1IuN1qlJcx1ykJ0OR9KyW7+ItXVqI6I+9817OP7L77Ak9a3vW4cGUAORIkNyHH1Os1+oVPxFYAYmkKzx03yuypItzu3m5hlS+ESQFM6G0t/ZOLqNRD4+QbElHfOgRnlh1WRVLLQBKglZLMnA9QG01zclDCWn1vohTsPkyt76AynPIS3gHjpp+hSSmtAgQKrqhVCpqKT8Buic2a509iFVASSsmiGK5kYRSUOFkHGoTMgq5rl2k+bZB8DecSLOc0MVWMY2fW0cqrePPNZYFE57XztJV/qpyZ+baph+nPIIGUtCia4oTT0ciV7/Q14wpJWKk7F+klOHjKKUUVSUWr9kuNMHjuREpa2eUPQ/u0IXTv5b3f+/+d5/l+rNUH+O2gfNGu78/8PSf5DUkx+PUpv9g7lwNvsP9ttS9l/NhstlP9atfou/v/7kv++gFnHG5wLh54dvnlL+pLwK7qGy4vFdDzzwhBPKzAcYDx0Zux+16r9NXCpEcbg8zE+/2XNPOALukKH/EpAEnRKESQE84ofeLEr6vXmJfzHW8WWDkieIIIGS5Ao4qHru3RZEQbjd4j/YMcV6oqRCofKH4zTCYNZPFyTBVH0u/XOrjJ/PUd7Rfr12J9iFP/SYKMz4JGI/Alf/s7DMuVhrA3pJJJ9s/H27UHmyMYdz+7f5gB2vf6q/2CuCLnfbfQhLB4lqpNEOQbzQTo1uhqlOfeiKIj03Ikjt3V93rtD3CRFjVo5ZjQMYKypTh2m2JDNmckUhHEfBFhdt6w68MRWHUc0yI40s8vGAD4mWntMQCBF2bveapWFXke/M/qDjOiIzuoBtgvIeShPLAYTk7CJfcSwBDCy0PiFN0qmwM+C7McVcfKUQkJyOnYGz4SyA9QscryEbXXVzbGqpc3GztJV0AlvNJ4rddJyvbd3jnfRK9OZG6OUca50IrB17jOkXWdhj0TmzISeje+cp/3z4P/q6/xfbcf/vRf+r13w/zlsNxvm3Xa9fbDj/j4o/i8ESuhErk272zs+A7iF/2s3rdT/x2q1mxj/wTrYxX94X/zfA3TiEI6I53hvEal5K6jL9NxiIAhimHJ+yGSniU7nyOOlrsiZJ7JhliriYRYCORa/r1RQq0RN/D519405NDLeCIXO1jps3eOQgzjnGC5WP0ONz1f4VOJKdTLPG+A1VftW66+B4cSwA2NmYmXvkAHlsmjWhLZ4IzQZZPWa0MfAaCXYRfR7gfTfS70bbP+qoiygReSKr5+/EOixDYND5heZU2VtFPMjGSzI6tHIBQNl0NZ+6Y4IOKfE5aYjwTLjBcxBsWuY88mLTmocC2xy7A4NsmWdusUg0+xKzaG1M3aZjRfJzsavkLZWVHOZ0mEBL3QVbzZ+zcJG82nKW16B+zMMUWm682aNuaAab/LqKubPc+kbDVNTRNzoF9XYbkeKQR42l6ltLiTxaWMZ4JBvcx4r5Mewu5tMPFcCXjgmjX8t7AVgz8agF70On6VgNWuSB1m0bI1dUlTkZtkzI+EVOSUAPjk+z0wg8ZwnRW19PHPQji8Yj6URFPHOscxOlp8506lx0XSKCmOgU7qLqZ8zo8KLddlYLbtdFyv1VkMs53Ipo7ROns/XoWfQAtpdSbc2TtA9kDcsQyWjqBErU9WVIKN0TubRLTkYQwelH1L7A+wNtFlMMVP2n0JOkCnbedxbep2lR+Z3abb+kVjiESCFHGG8v4Y81Wo+U6dWH3HQERkVBHt2RJ3PjNLIyNAlIZIazeD3KtwSgiTFtPVAJO55PhSJ499WQy6eRbGK0J9oOZM6JGaEECAZsjCZk15DIxe+2PEBNoVjTjSsRIHS9wA/adOgsxo8DLmu3WRtnBejmbwKMdeR+KIsvkcBMgtkkpbg3BzCY20V8FBs0moQRMriPGvszA1R5mdscExJLxgrXoUIPF+ejWLO3H1Z56n0mQqeY217kxqkaHi30vmaV0eHUQjzkMVjBzEDntDAsaNQ3C+g+E3/pmihGXgUoAngXTRfxHP3y5i1JLTppHBHq0sotRZDHod6GaPRebNW23BnNX4uYdlcWmVxBd9XFtR+icoMH8/N8RfeQg0v0rQrmQYC9WVZwK7Ft1abVkvsCf3SgqFd1gwSvdPUK0y9yoe/eduJhm33GgZ2o+a6h21jR2uV8MroYJ++wG7vwxPdTI8vL+HlJb/8Hse2D0/9lbu/iMAB6g+CYKZLOHopHE08QjTEPeDParwMiMwg1vSL0cbfAnNWxlBAnsw5IH+kfMnHyV8A7L6HYTWN8nqUb/7cin1xinvuDZMl2EcIuzbtFKuYmI66iIjjtUXt9ddwb5wLUvRX+SBFnbS8DFI03hwiSdZzkbvHfWMYptWdtXpNQLjR3iI8k9x6tjWfBl4a54L04xqGNaiWb2ktnFJOX3VezrESefwuqCs3aKuyOuLtdUiatPnMlvVdUCjdE29o50Nx4LoY04eq3b+TWqbcMSAvPxxx3uAcUoqK7TK/ifFNnOe/P8wIQDv73539b0H/Z7Uw/uvhgbVTAH5I+j/SVfwit7+9Rfyfg1qm/2vg+q83D3b3v72Xz7b4P2w7+8G4NDMT/LYOzXltE7kw5yP0sM0cuQehaaJGBnWQZ0tsnrONep7WFlfeeLPf7aq3JMVgJMNb5a+YE5GhGN11vHKoqwL24PeZVD2kzFjUu0OM1Z3+zcsEn4hRv9OnO9IqmADcunyMhU6G3pjK/nLqhfEy4SvnqQb8hW8adPuXakp64l+Tf1feXrGSGB2zNr4R8w82WONfLP+3s//71fi/ov2f1TywzLp12G7XdvEfPyT+L1r46EJh45kfbBrTd73+33D/i9Woq/t/WwfWwQGd/7abO/7vfXw+/qvqwPOrAyeelj4Wn1PMZTr3nTtJ5F3yTbpsbs8h18UROirMnehK4K3BFJNA5UAkGmMV5iyYlNxL4uBenNz/+vTz5yfPHp+c2t89Pnnw/PTpi991XbQ7K7lL3bgWLnBPQut2xSeW+KQuPmloRyK8SqboObYen0pIzo5Ph73RpQ3F0qg0YvX4L1U7fVLnAXzSEHRmmHb7k2vrBt5+0uT39Xu/scRrMYncUFQeC+1v2d/hNfNzr5GZM6CDN6WeqIyzThS8garoKs7V9cXr16ujKUZbFSqYy3rfZTr3d3NTQoarFG/rlyToeFEgzJTPDcZAKYybaiQ20qc7EgsN8nWEFbYX2BgS80iMAmAg3aX4xBc5K7ejQpLIRebBM6iWJmyqFIr2MFt3peG++M1vxHWhDlUjqfJwUkZ44raj6zv+b8f//Qz+r9U8ODQPD5r1uzsLwA+O/yM/33fM+70N/9dst7P4j032/7faO/7v/fN/dBVUJXYT5gH19FImA93tge9D1/SOiM88YCDIrgdDAbBJG5rEMEfFFxKbb8MAHokH3c3+5jnu6isbmYyfyVJ98mADE8U1ppzTm6J757gkjL1d6BZzJ++2Y1znz2Hq8swcsW7f0C3PGfeVskzEnNFAFAvMALE/+Wb7OLZy4CkwFQcOI0t5blmhDAQpjSGL7Hex7fWeSZ/2n9m3FJ5v7F2xP8UWmaUkGeX+V189ev7147+gnXHH/+34vxz/12y22qbVbrUazR3/9yHxfxw5I1yYtO29+/X/Jv+Per2u+L9WDdOtRsvanf++l8/IG49h/5t4iXAICarEd1RH7hC+I9xQB5vTS7zN1u86zqjhmmbbGo5Gg7qwaLJLlUplW42l/f39rbV+9pmoNGq18oHY5y9IGM6cOBYvIsePga+Zu9Ejzq/7vvksGC1m7krETAqZ6cbBbEHhp3LG3GzenQCL5BZsvNctFceuk9ho6EchZdO6qlVZfL3EMAiiUWxPy+rXBVltz8amjdGaOFEvVTZZRKatlfM/OcBTl6MIlfZ/ckGdSyI4bJtP1G1bl1GhjE1Rp/hKQw0ogWasj3BDEo1vGMxDrgFGGY04FhAMXl8HyVtUEUP/3JGCvb5phEZpG+KGQezhRNkgSgQYqWwNhTflkMg8bhwO2/WBaR4etF234W5F5o11rKH1xlyI4K064jf9TdH7WGZ+rEKsnXq+ux3D8aODGIBwWySuXUTRRr0s3vBy1TK3aB3MBrcIfc/P14JJcWcdfxPXj4MoTuOT/cGNgljXLYyKhL4S66hsrOPyT6nkX4fW2zAQRn0BUpIue7Ih28divIiSqRvBlClaMgLhbzjFqwyUPxbhvv4KNhb0Cxu4InbG7la0x9w9HF1fEYtckknt6IbJrejb0T4L/2SHUTB0YxjCGuZvyaSQ/7DZuDtumWZ9OByNW+OtyL+tmjX835YRl4DVxCVAf+GRLybF+Vt6GKDOTBSxTwOCLesluVJOodpjVZ9cFpqm4f+SYPREpyJADd9DxECocqzuWTlHyrswvsMVxEQ65Y+BuvtD105vwaIrBzqMsz+95n8dtr6xR0VcYo1BVx7eFN/lKEB+O1vJxT3Gcxf6sRXbljOg8/MBEKh1JCu+k7g1rB2M3buWaYKQMRq0727FrZXSayi18p4wqU7UlL9ScorB3r776oEzPBsE2+ho5pQF9A1QR6gJlqTAxugjuQ2bp3vohAlfY+b5UM6mcGldDgjvAOXGl86MysrULdv4v7K6d7G5b4AGhuxckhsWRdKzLyJ0FYl0ZmTykLH9wAb4UjZDLxI4NapuOjyeq8YBzRV9/aS5yurJmLxVeGWvNgIum8jims9m5N2s2HXQMobxNReCWdx6m1hc+koB8R0Rv+f+7OovEAiDWHqxqVEYmwC00zrs9H87/d+vpf+z2pZlNtp32+3Dnf7vQ9T/IaM+eMe3AN7m/3HQaKT2f02rgfZ/9ebu/r/38gGZ6RhnXMZwJ2ENZVp3mKBLty8eHn/bERi8jhwMQBIHXrDCx3gygXgP0hlQsD5AoDlgVD6MCIdhJ88SihvCEdg3RwyhVMRGFmdsOj7FuCn5y6JzgqW0VCuWMjdJnqlLSV58XI8tjWHzNreipxwTcjhHgq73ylemz3O8PuZB5knjFA31A4XcphIj4QeqM+HHMLCXDnnRxMxMrVSv3CQ0GZC6vD0cdVmMZ4t4Kh1ZhuQ+k/oVPwQObBG567HelS8tJ9VqHOz9CBlPE2MnYA0P7x/beJOWffz81H52+vghXqZVw+AnwVmZnaQhNwcYKeUvNMui/2E1w2XyMJjBqMbs8Pzw+VfPT+wHT07qJ08eGOzDQlEcehqfW2sUazqSbsOODzMz0/rMYK5HCMeUY+wzz6AOHUFHGWs1I8db5Jwk1DD+6vzVDQFvMXZQN5aBF3gCQnKIl3XnJ8LK4K9xFdunyVJ3FWkc6wG60tPILSamW5p6XCrGONvZBWCFXHjDlhdTBJdWv1+c839D/N/O//dX4/92/r8f9McJw1/I6/et+b9GIz3/Vf6/tWZ9F//vffF/TyJn5AXAuMyDjjKsYrRY8bTAnxwgcmuIOGLvJlyhg6ZopV/bszcNW/YWeeOpE6HrLt9a1NXoINJOIP+2q3U33oNLIc2IRbadRKcOlJEpABSS+rAt7Fjhstfs8lIOBnY7F5aIPdSrb+bEjuRvFe0lF/MsOEtv2EnjlcEr6G5HxrsZJo4/mVHIm2QRwg94hxFyjPxzvYPPaTDnGl0nkg9+djvXR5CLFj5ya+p2VeWLjHeXAjOZBtWRlcoocpvuaIXsaz7OXCeycGfyIHDi4JlJT08nLLsBtYx3m6orP+kBxkms2Vi7jjOXaPGZwMeis7PQMS31cq6PbwzNKN7jyt2OgguK6dRbaVjePVpOL1hd6Y7kDiUziCkNo7+xAYYUDLRMjZXomulJZD6YBcOzGNjSZAYSy7/8lx9/+A/Cqos//d0/iBf//L/m4sz78Y//dy5+/OM/+tPqv/z9jz/8vSeS//e/f/zhH/ECsCiAv9xhAxc7ERAZsc585kRnGB1e1z7+eL0yURVvqE786T/9V/nLnyx+/OEffLw3d/rjD//Ng2Lejz/8x/lL/wVU5Mna9Gnw4x//z1A+UUDQleoNMfvn/+mb8thDjf8kuNDzQQJJVoF0Eld08srXQg/vIZ85A3fW1X7843+HBoAgXInlj3/8Jx/JQlpc1Qr4vZj7+sqBPd2nTLWfIHXU3xAooIyhVRduGltANv4QIPCfxfzHH/5piO2ecW2nMw8tYSy+0biF4oobdq201Au87h7zDxKfSzxYJEkAE4OzouXXALx74sxmbnSly8JfwpRh5CQALB2N4sBiCk6QOAMu8Qi2BhqJPiURAkMgqLiajMJ6bGgczkv+ohuC+KdEXRljPfHN4cwbnulAAsp4p1hGBHCVEwJDw32kE/PAnDkLEFl1otpdx6TvXWSAnf5/J//9XP1/665Vt8za3bv1u436biV9AB/JFdPFnxga+FeQ//ACWCX/1Vt1C+M/tVo7/f97+XwsElfdzUMMP+nmRd20mqa1PwwXsONmdrexaJn1mgky3tjx4rgC74VlWi3T4mCPw2WFBcgKMgMzvASoZdbg391GKha2zQbGG6Z27nWhoVK+gXvdptlsldLq73Ut87C0uW7Me1iiIOQlYNNmwUWJ28AXtR352n12n91n99l9dp9tn/8P9qnzuQAYAQA='
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE)), mode='r:gz').extractall('.')
sys.path.insert(0, '.'); os.makedirs('results', exist_ok=True)
from vsearch.weights import discover
t=time.time(); W = discover('/kaggle/input', '/kaggle/working/w'); W['assemble_s']=round(time.time()-t,1)
json.dump(W, open('results/weights_layout.json','w'), indent=1)
ok = W.get('dinov3') and W.get('sam31_ckpt') and W.get('sam31_sha256_ok', True)
if not ok:
    open('results/STATUS.txt','w').write('weights not found / sha mismatch: '+json.dumps({k:v for k,v in W.items() if k!='layout'})+'\n')
    os.makedirs('/kaggle/working/results', exist_ok=True); os.system('cp results/* /kaggle/working/results/')
    raise SystemExit('stop: weights missing')
os.environ['DINOV3_PATH']=W['dinov3']; os.environ['SAM3_CKPT']=W['sam31_ckpt']
print({k:v for k,v in W.items() if k!='layout'})
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install faiss-cpu
!git clone -q https://github.com/facebookresearch/sam3.git /kaggle/working/sam3 && pip -q install -e /kaggle/working/sam3 && pip -q install pycocotools psutil
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
!wget -q https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip -O /kaggle/working/davis.zip && unzip -q /kaggle/working/davis.zip -d data && rm /kaggle/working/davis.zip
!python scripts/prepare_davis.py && python scripts/make_hard_queries.py && python scripts/add_distractors.py

In [ ]:
# 1) VRAM & tốc độ thật: DINOv3 embed, SAM 3.1(detector) trên keyframe
!python scripts/gpu_bench.py 2>&1 | tail -40

In [ ]:
# 2) Index 70 video: DINOv3 frame + grid region ; DINOv3 frame + SAM3.1 region
B='data/davis_bench_large'; D=os.environ['DINOV3_PATH']
!python scripts/build_index.py --videos {B}/videos --out data/L_dinov3 --model {D} --regions grid 2>&1 | tail -2
!python scripts/build_index.py --videos {B}/videos --out data/L_dinov3_sam3 --model {D} --regions sam3 2>&1 | tail -2

In [ ]:
# 3) Đánh giá (R@1, mAP@10): frame / region / both ; + SAM 3.1 re-rank top-20
for Q in ['object_hard','object']:
    !python scripts/evaluate.py --index data/L_dinov3 --queries {B}/queries_{Q}.json --out results/K_dinov3grid_{Q}.json 2>&1 | grep -E '^(frame|region|both)'
    !python scripts/evaluate.py --index data/L_dinov3_sam3 --queries {B}/queries_{Q}.json --out results/K_dinov3sam3_{Q}.json 2>&1 | grep -E '^(frame|region|both)'
    !python scripts/evaluate.py --index data/L_dinov3 --queries {B}/queries_{Q}.json --modes both --rerank sam3 --videos {B}/videos --out results/K_dinov3_rerank_{Q}.json 2>&1 | grep -E '^(frame|region|both)'

In [ ]:
!for d in data/L_*; do python -c "import json,sys;d=json.load(open('$d/meta.json'));d.pop('meta');print('$d',d)"; done | tee results/index_stats.txt
!mkdir -p /kaggle/working/results && cp -r results/* /kaggle/working/results/ && ls /kaggle/working/results